### Running the Experiment

This section describes the experimental pipeline used to train and evaluate a model for classifying source code as either human-written or machine-generated. The workflow covers data preparation, augmentation, model fine-tuning, and performance evaluation.

Our approach is based on fine-tuning CodeBERT on a combination of original training data and a large augmented dataset. To improve robustness and generalization, the training setup includes several optimization and regularization strategies, such as partial layer freezing, differentiated learning rates across model components, dropout, and label smoothing.

The experiment is organized into a sequence of clearly defined steps, presented below.

The entire process is organized into a sequence of clear steps, outlined below.

### 1. Storage Setup

Configures persistent storage for saving model checkpoints and training artifacts.

Google Drive is mounted to ensure that all generated files (e.g., model weights, checkpoints, and results) are preserved across sessions. The root directory of MyDrive is defined as the base path for managing experiment outputs.

This setup is particularly important when running in environments with ephemeral storage, such as Google Colab, ensuring data persistence throughout the project.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_BASE = "/content/drive/MyDrive"
print("Drive mounted:", DRIVE_BASE)


### 2. Environment Setup

Installs the required libraries for model training, dataset handling, and evaluation. This step needs to be executed once.

In [ ]:
!pip -q install -U datasets transformers accelerate scikit-learn scipy


### 3. Environment Configuration and Dependencies

Configures the environment for model training, including core dependencies, reproducibility settings, and evaluation utilities.

Core Imports: Loads system utilities (`os`, `re`, `json`), numerical computing libraries (`NumPy`, `PyTorch`), and the Hugging Face ecosystem (`transformers`, `datasets`) for natural language processing tasks.

Reproducibility: Sets a global random seed (42) across all relevant frameworks to ensure consistent results (e.g., model initialization, data ordering) across runs.

Evaluation: Imports performance metrics (`f1_score`, `confusion_matrix`) and post-processing functions (`softmax`) for analyzing model outputs.

In [ ]:
import os, re, glob, math, json, random, hashlib
import numpy as np
import torch

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
)
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy.special import softmax

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("seed:", SEED)


### 4. Configuration and Hyperparameters

Defines the experimental configuration, centralizing all key hyperparameters and training settings.

Model Core: The microsoft/codebert-base model is used as the backbone, optimized for source code understanding tasks.

Training Setup: Specifies the training regime, including the number of epochs, batch sizes, sequence length, and gradient accumulation strategy, along with regularization techniques such as dropout and label smoothing.

Optimization: Implements differential learning rates across model components, assigning distinct learning rates to early encoder layers, later encoder layers, and the classification head. It also applies partial freezing (first N layers and embeddings) to stabilize fine-tuning.

Persistence: Defines the output directory (RUN_DIR) for storing checkpoints, logs, and training artifacts.

Scheduling and Constraints: Includes training control parameters such as warmup ratio, weight decay, gradient clipping, and evaluation/save intervals.

Data Scaling: Configures dataset-related parameters, including target training size for augmentation and out-of-distribution validation limits.

In [ ]:
SUBTASK = "A"
MODEL_NAME_OR_PATH = "microsoft/codebert-base"
MAX_LEN = 384
DROPOUT = 0.20
LABEL_SMOOTH = 0.08
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
EPOCHS = 2
BSZ_TRAIN = 32
BSZ_EVAL = 64
GRAD_ACC = 1
FREEZE_EMBEDDINGS = True
FREEZE_FIRST_N_LAYERS = 6
LR_EARLY = 5e-7
LR_LATE  = 8e-6
LR_HEAD  = 1.5e-5
RUN_DIR = f"{DRIVE_BASE}/semeval_codebert_full500k_plus_bigaug_v3"
os.makedirs(RUN_DIR, exist_ok=True)

SAVE_STEPS = 1200
EVAL_STEPS = 600
SAVE_TOTAL_LIMIT = 3

OOD_DEV_TOTAL = 8000
AUG_TARGET_TRAIN = 500_000

print("RUN_DIR:", RUN_DIR)


### 5. Augmentation Data Configuration and Validation

This section defines the external augmentation datasets used to enrich the training data and validates their availability.

Augmentation Sources: A set of JSONL files is specified, representing diverse augmentation strategies, including machine-generated transformations (e.g., translation, domain shift) and human-like augmentations generated via prompting. These datasets are used to increase data diversity and improve model generalization.

LLM-as-Human Data: Additional datasets can be optionally included as synthetic samples intended to approximate human-written coding style using LLM-generated outputs.

File Validation: All specified file paths are checked for existence. Available files are collected and reported, while missing files are safely ignored to ensure robustness and prevent runtime failures.

Logging: The script outputs a summary of detected files, clearly distinguishing between available and missing resources.

In [ ]:
AUG_JSONL_FILES = [
    f"{DRIVE_BASE}/machine_translate_unseen_2400.jsonl",
    f"{DRIVE_BASE}/machine_domain_shift_1500.jsonl",
    f"{DRIVE_BASE}/human_simple_aug_3000.jsonl",
    f"{DRIVE_BASE}/human_domain_aug_2000.jsonl",
]

LLM_AS_HUMAN_FILES = [
    # f"{DRIVE_BASE}/human_translate_student_750.jsonl",
    # f"{DRIVE_BASE}/human_translate_student_750_v2.jsonl",
]
LLM_AS_HUMAN_CAP_TOTAL = 200

existing, missing = [], []
for p in AUG_JSONL_FILES + LLM_AS_HUMAN_FILES:
    if os.path.exists(p):
        existing.append(p)
    else:
        missing.append(p)

print("Existing JSONL:", len(existing))
for p in existing:
    print("  ✅", os.path.basename(p))

if missing:
    print("Missing (ignored):", len(missing))
    for p in missing:
        print("  ⚠️", os.path.basename(p))


### 6. Dataset Loading and Split Initialization

Loads the SemEval-2026 Task 13 dataset and initializes the predefined data splits used throughout the experiments.

Dataset Source: The dataset is loaded using the Hugging Face datasets library, ensuring standardized access and compatibility with the training pipeline.

Data Splits: The dataset is divided into three official subsets:
- training set (train)
- validation set (validation)
- test set (test)

Each split is assigned to a separate variable for clarity and modular processing during training and evaluation.

Inspection: Basic information about each split is printed, including the number of samples and the available columns. This step helps verify data integrity and ensures that the dataset structure matches expectations before further processing.

In [ ]:
base = load_dataset("DaniilOr/SemEval-2026-Task13", SUBTASK)

train_base = base["train"]
val_official = base["validation"]
test_official = base["test"]

print("train_base:", len(train_base), "| cols:", train_base.column_names)
print("val_official:", len(val_official), "| cols:", val_official.column_names)
print("test_official:", len(test_official), "| cols:", test_official.column_names)


### 7. Helper Utilities for Normalization, Checkpointing, and Evaluation

Handles code preprocessing, hashing, checkpoint selection, and evaluation metric computation used throughout the training pipeline.

Normalization: Standardizes code strings by collapsing multiple spaces/tabs into a single space and normalizing line breaks to prevent formatting-based variance.

Hashing: Generates a consistent MD5 hash of the normalized code, enabling robust identification and deduplication regardless of minor formatting differences.

Checkpoint Management: Resolves the optimal model path by prioritizing the "best" checkpoint from the trainer state, falling back to the most recent one if necessary.

Evaluation: Computes the macro F1 score by converting model logits into discrete class predictions and comparing them against ground truth labels.

In [ ]:
def norm_code(s: str) -> str:
    s = (s or "").replace("\r\n", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def md5_code(s: str) -> str:
    return hashlib.md5(norm_code(s).encode("utf-8", errors="ignore")).hexdigest()

def latest_checkpoint(path: str):
    cks = sorted(
        glob.glob(os.path.join(path, "checkpoint-*")),
        key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else -1
    )
    return cks[-1] if cks else None

def best_or_latest_checkpoint(run_dir: str):
    ts = os.path.join(run_dir, "trainer_state.json")
    if os.path.exists(ts):
        with open(ts, "r", encoding="utf-8") as f:
            st = json.load(f)
        best = st.get("best_model_checkpoint", None)
        if best and os.path.exists(best):
            return best
    return latest_checkpoint(run_dir)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}


### 8. Augmented Dataset Loading and Preprocessing

Loads and preprocesses augmented datasets from JSONL files, ensuring consistency, basic filtering of empty or invalid samples, and controlled integration of additional data sources.

Data Loading: Reads JSONL files using the Hugging Face datasets library and retains only relevant columns (e.g., code, label, language, domain metadata).

Filtering: Removes samples with missing or empty code fields to ensure data quality and prevent invalid inputs.

Normalization: Casts all code entries to strings and labels to integers for consistent downstream processing.

Augmentation Aggregation: Iteratively loads and combines multiple augmentation sources into a unified dataset.

LLM-as-Human Integration: Optionally includes synthetic data generated by LLMs, applying a global cap and random sampling to prevent overrepresentation.

Final Dataset: Concatenates all valid augmentation parts and shuffles the result to ensure randomness before training.

In [ ]:
from datasets import load_dataset as hf_load_dataset

def load_jsonl_min(path: str):
    ds_aug = hf_load_dataset("json", data_files=path, split="train")
    keep = [c for c in ["code", "label", "language", "target_domain", "source_language", "generator"] if c in ds_aug.column_names]
    ds_aug = ds_aug.remove_columns([c for c in ds_aug.column_names if c not in keep])

    ds_aug = ds_aug.filter(lambda x: x.get("code") is not None and len(str(x["code"]).strip()) > 0)
    ds_aug = ds_aug.map(lambda x: {"code": str(x["code"]), "label": int(x["label"])})
    return ds_aug

aug_parts = []
for p in existing:
    d = load_jsonl_min(p)
    aug_parts.append(d)
    print("✅ Loaded", os.path.basename(p), "rows:", len(d))
llm_parts = []
for p in LLM_AS_HUMAN_FILES:
    if os.path.exists(p):
        d = load_jsonl_min(p)
        llm_parts.append(d)
        print("⚠️ Loaded LLM-as-human", os.path.basename(p), "rows:", len(d))

if llm_parts:
    llm_all = concatenate_datasets(llm_parts)
    if len(llm_all) > LLM_AS_HUMAN_CAP_TOTAL:
        llm_all = llm_all.shuffle(seed=SEED).select(range(LLM_AS_HUMAN_CAP_TOTAL))
    aug_parts.append(llm_all)
    print("⚠️ Using LLM-as-human capped:", len(llm_all))

if not aug_parts:
    raise ValueError("No augmentation files loaded. Check paths in AUG_JSONL_FILES.")

aug_all = concatenate_datasets(aug_parts).shuffle(seed=SEED)
print("Aug total loaded:", len(aug_all))


### 9. OOD Split and Training Partitioning

Splits the augmented dataset into a development split derived exclusively from augmented data (referred to as OOD dev) and the remaining training data.

Adaptive OOD Sizing: Computes the validation set size as 20% of the total data, with a minimum of 1,000 samples and a maximum global cap (OOD_DEV_TOTAL).

Dataset Partitioning: Splits the dataset into two subsets: the OOD development set and the remaining training pool.

Shuffling: Applies reproducible random shuffling with a fixed seed to ensure both subsets are well mixed.

Verification: Logs the final sample counts to confirm that the partitioning meets the expected constraints.

In [ ]:

dev_n = min(OOD_DEV_TOTAL, max(1000, len(aug_all)//5))
ood_dev = aug_all.select(range(dev_n)).shuffle(seed=SEED)
aug_train_rest = aug_all.select(range(dev_n, len(aug_all))).shuffle(seed=SEED)
print("OOD dev:", len(ood_dev))
print("Aug train rest:", len(aug_train_rest))


### 10. Dataset Upsampling

Ensures a consistent training volume by normalizing the dataset to a fixed target size.

Dynamic Scaling: Automatically performs downsampling (if too large) or upsampling (if too small) to reach AUG_TARGET_TRAIN.

Stochastic Repetition: During upsampling, uses varying seeds across repeated blocks to maintain data variety.

Fixed Output: Guarantees that the final aug_big dataset contains exactly the required number of samples for the training pipeline.

In [ ]:
def upsample_to(ds, target_n: int, seed: int = 42):
    if target_n is None or target_n <= 0:
        return ds
    if len(ds) == 0:
        return ds
    if len(ds) >= target_n:
        return ds.shuffle(seed=seed).select(range(target_n))
    reps = (target_n + len(ds) - 1) // len(ds)
    parts = []
    for r in range(reps):
        parts.append(ds.shuffle(seed=seed + r))
    big = concatenate_datasets(parts)
    return big.select(range(target_n))

aug_big = upsample_to(aug_train_rest, AUG_TARGET_TRAIN, seed=SEED+123)
print("Aug after upsample:", len(aug_big))


### 11. Data Leakage Prevention

Ensures the integrity of the evaluation by removing any training or OOD samples that overlap with the official validation set.

- Content Hashing: Generates MD5 hashes for all samples in the official validation set to create a unique fingerprint lookup.

- Validation Isolation: Filters both `aug_big` and `ood_dev` to exclude any entries that match these hashes, preventing data leakage.

- Leakage Verification: Logs the final counts to confirm how many overlapping samples were removed before proceeding.

This step is critical for preventing evaluation leakage and ensuring a fair and unbiased comparison with the official validation set.

In [ ]:
val_hashes = set(md5_code(c) for c in val_official["code"])
print("val hashes:", len(val_hashes))

def not_in_val(ex):
    return md5_code(ex["code"]) not in val_hashes
aug_big = aug_big.filter(not_in_val)
ood_dev = ood_dev.filter(not_in_val)

print("Aug after de-dupe vs val:", len(aug_big))
print("OOD dev after de-dupe vs val:", len(ood_dev))


### 12. Final Dataset Consolidation

Creates the final training set by merging all processed data sources.

- Dataset Integration: Combines the base training dataset with the augmented dataset (`aug_big`) into a single `train_full` dataset using `concatenate_datasets`.

- Stochastic Mixing: Applies a global shuffle to intermix original and augmented samples.

- Inventory Logging: Prints the final counts of all partitions to verify the integrity of the training pipeline.

In [ ]:

train_id_full = train_base
train_full = concatenate_datasets([train_id_full, aug_big]).shuffle(seed=SEED)
print("Train ID full:", len(train_id_full))
print("Train aug big:", len(aug_big))
print("Train FULL:", len(train_full))
print("OOD dev:", len(ood_dev))


### 13. Tokenization and Formatting

Prepares the datasets for model training by converting raw code into tokenized inputs and aligning feature structure.

- Tokenization: Applies the pretrained tokenizer to the `code` field with truncation up to `MAX_LEN`, without padding at this stage.

- Batch Processing: Uses batched mapping for efficient tokenization of both training and validation datasets.

- Feature Selection: Retains only the required columns (`input_ids`, `attention_mask`, `label`) by removing all others.

- Dynamic Padding: Initializes a `DataCollatorWithPadding` to handle padding dynamically during training.

- Dataset Verification: Logs the final sizes of the tokenized training and validation sets.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, use_fast=True)

def preprocess(batch):
    return tokenizer(batch["code"], truncation=True, max_length=MAX_LEN, padding=False)
train_tok = train_full.map(preprocess, batched=True)
dev_tok   = ood_dev.map(preprocess, batched=True)
keep_cols = ["input_ids", "attention_mask", "label"]
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep_cols])
dev_tok   = dev_tok.remove_columns([c for c in dev_tok.column_names if c not in keep_cols])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenized train/dev:", len(train_tok), len(dev_tok))


### 14. Model Configuration, Freezing, and Layer-wise Optimization

Configures the pretrained classifier for binary prediction, applies selective regularization settings, and defines a layer-wise learning rate strategy for fine-tuning.

- Model Configuration: Loads the pretrained configuration with `num_labels=2` and updates supported dropout attributes to use the specified `DROPOUT` value.

- Backbone Detection: Identifies the transformer backbone (`RoBERTa` or `BERT`) in order to access its embedding and encoder layers programmatically.

- Selective Freezing: Optionally freezes the embedding layer and the first `FREEZE_FIRST_N_LAYERS` encoder layers to reduce overfitting and preserve lower-level pretrained representations.

- Weight Decay Handling: Separates parameters subject to weight decay from biases and normalization weights, ensuring regularization is applied only where appropriate.

- Layer-wise Learning Rates: Builds optimizer parameter groups with different learning rates for the classification head, earlier encoder layers, and later encoder layers, enabling more controlled fine-tuning across the network depth.

- Custom Optimizer Setup: Overrides the trainer optimizer creation step to initialize `AdamW` with the custom layer-wise parameter groups.

In [ ]:

config = AutoConfig.from_pretrained(MODEL_NAME_OR_PATH, num_labels=2)
if hasattr(config, "hidden_dropout_prob"):
    config.hidden_dropout_prob = DROPOUT
if hasattr(config, "attention_probs_dropout_prob"):
    config.attention_probs_dropout_prob = DROPOUT
if hasattr(config, "dropout"):
    config.dropout = DROPOUT
if hasattr(config, "attention_dropout"):
    config.attention_dropout = DROPOUT

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_OR_PATH, config=config)
if hasattr(model, "roberta"):
    backbone = model.roberta
    layer_prefix = "roberta.encoder.layer."
    emb_module = backbone.embeddings
elif hasattr(model, "bert"):
    backbone = model.bert
    layer_prefix = "bert.encoder.layer."
    emb_module = backbone.embeddings
else:
    backbone = None
    layer_prefix = None
    emb_module = None

if emb_module is not None and FREEZE_EMBEDDINGS:
    for p in emb_module.parameters():
        p.requires_grad = False

if backbone is not None and hasattr(backbone, "encoder") and hasattr(backbone.encoder, "layer"):
    n_layers = len(backbone.encoder.layer)
    n_freeze = min(FREEZE_FIRST_N_LAYERS, n_layers)
    for i in range(n_freeze):
        for p in backbone.encoder.layer[i].parameters():
            p.requires_grad = False
    print(f"✅ Frozen embeddings={FREEZE_EMBEDDINGS} and first {n_freeze}/{n_layers} layers")
else:
    print("⚠️ Could not locate encoder layers; skipping freezing.")

no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

def add_group(param_groups, named_params, lr):
    if not named_params:
        return
    decay_params, nodecay_params = [], []
    for n, p in named_params:
        if not p.requires_grad:
            continue
        if any(nd in n for nd in no_decay):
            nodecay_params.append(p)
        else:
            decay_params.append(p)
    if decay_params:
        param_groups.append({"params": decay_params, "lr": lr, "weight_decay": WEIGHT_DECAY})
    if nodecay_params:
        param_groups.append({"params": nodecay_params, "lr": lr, "weight_decay": 0.0})

def build_param_groups(model):
    groups = []
    head = [(n, p) for n, p in model.named_parameters() if "classifier" in n]
    add_group(groups, head, LR_HEAD)

    if layer_prefix is not None and backbone is not None and hasattr(backbone, "encoder"):
        n_layers = len(backbone.encoder.layer)
        cut = int(math.floor(n_layers * 2 / 3))
        early, late, other = [], [], []
        for n, p in model.named_parameters():
            if "classifier" in n:
                continue
            if n.startswith(layer_prefix):
                parts = n.split(".")
                idx = None
                try:
                    idx = int(parts[3])
                except Exception:
                    idx = None
                if idx is not None and idx < cut:
                    early.append((n, p))
                else:
                    late.append((n, p))
            else:
                other.append((n, p))
        add_group(groups, early, LR_EARLY)
        add_group(groups, late + other, LR_LATE)
    else:
        rest = [(n, p) for n, p in model.named_parameters() if "classifier" not in n]
        add_group(groups, rest, LR_LATE)

    return groups

class LayerwiseLRTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer
        from torch.optim import AdamW
        self.optimizer = AdamW(build_param_groups(self.model))
        return self.optimizer


### 15. Training Setup and Execution

Defines the training configuration, enables checkpoint resumption, and initializes the custom trainer with evaluation and early stopping.

- Checkpoint Resumption: Detects the latest checkpoint in `RUN_DIR` to allow training to resume from a previous state if available.

- Training Configuration: Sets up `TrainingArguments` including batch sizes, gradient accumulation, number of epochs, gradient clipping, and mixed precision (`bf16`).

- Evaluation Strategy: Performs evaluation and checkpoint saving at regular step intervals, with `f1_macro` selected as the metric for model selection.

- Learning Schedule: Uses a linear learning rate scheduler with warmup and applies weight decay and label smoothing for regularization.

- Early Stopping: Configures an `EarlyStoppingCallback` to halt training when performance stops improving based on validation metrics.

- Trainer Initialization: Instantiates the custom `LayerwiseLRTrainer` with the model, tokenized datasets, data collator, metric computation function, and callbacks.

In [ ]:
resume_ckpt = latest_checkpoint(RUN_DIR)
print("Resume checkpoint:", resume_ckpt)

args = TrainingArguments(
    output_dir=RUN_DIR,
    seed=SEED,

    per_device_train_batch_size=BSZ_TRAIN,
    per_device_eval_batch_size=BSZ_EVAL,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    lr_scheduler_type="linear",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    label_smoothing_factor=LABEL_SMOOTH,
    logging_steps=50,
    report_to="none",
    fp16=False,
    bf16=True,
    dataloader_num_workers=2,
)

early_cb = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0005
)

trainer = LayerwiseLRTrainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    compute_metrics=compute_metrics,
    callbacks=[early_cb],
)


### 16. Training Execution and Evaluation

Runs the training process, optionally resuming from a checkpoint, and reports the best model along with final evaluation metrics.

- Training Execution: Starts the training loop using `trainer.train()`, resuming from `resume_ckpt` if available.

- Best Model Tracking: Retrieves and prints the path to the best checkpoint selected during training based on the configured evaluation metric.

- Final Evaluation: Evaluates the best model on the validation (`dev_tok`) set and logs the resulting performance metrics.

In [ ]:
trainer.train(resume_from_checkpoint=resume_ckpt)

print("✅ Best checkpoint:", trainer.state.best_model_checkpoint)
print("✅ Best OOD-dev metrics:", trainer.evaluate(dev_tok))


### 17. Official Validation Evaluation

Prepares the official validation set using the same preprocessing pipeline and evaluates the trained model on it.

- Tokenization: Applies the pretrained tokenizer to the `val_official` dataset using the same `preprocess` function as for training.

- Feature Alignment: Removes all non-essential columns, keeping only the required inputs (`input_ids`, `attention_mask`, `label`).

- Final Evaluation: Runs `trainer.evaluate()` on the processed validation set and logs the official validation metrics.

In [ ]:
val_tok = val_official.map(preprocess, batched=True)
val_tok = val_tok.remove_columns([c for c in val_tok.column_names if c not in keep_cols])

print("Official validation metrics:", trainer.evaluate(val_tok))


### 18. Test Inference and Evaluation

Generates predictions on the test set and computes evaluation metrics when ground-truth labels are available.

- Tokenization: Applies the same preprocessing pipeline to the `test_official` dataset.

- Flexible Feature Handling: Retains required input features (`input_ids`, `attention_mask`) and conditionally includes `label` if present.

- Prediction Generation: Uses `trainer.predict()` to obtain logits, converts them to probabilities via softmax, and derives final class predictions.

- Conditional Evaluation:
  - If labels are available, computes Macro F1 score, prints a detailed classification report, and displays the confusion matrix.  
  - If labels are absent, outputs a sample of predicted labels for inspection.

In [ ]:
test_tok = test_official.map(preprocess, batched=True)
keep_test_cols = ["input_ids", "attention_mask"]
if "label" in test_official.column_names:
    keep_test_cols.append("label")
test_tok = test_tok.remove_columns([c for c in test_tok.column_names if c not in keep_test_cols])

pred = trainer.predict(test_tok)
logits = pred.predictions
probs = softmax(logits, axis=-1)
y_hat = np.argmax(probs, axis=-1)

if "label" in test_official.column_names:
    y_true = np.array(test_official["label"])
    print("✅ TEST Macro F1:", f1_score(y_true, y_hat, average="macro"))
    print(classification_report(y_true, y_hat, target_names=["human(0)", "machine(1)"], digits=6))
    print("Confusion:\n", confusion_matrix(y_true, y_hat, labels=[0,1]))
else:
    print("ℹ️ Test has no labels. First 30 preds:", y_hat[:30])


Expected Test Performance

Running the evaluation cell above typically yields a Macro F1 score of approximately 0.40, although results may vary depending on factors such as randomness, hardware, and data shuffling.

The classification report and confusion matrix generally indicate an imbalance in predictions, with higher recall for the machine-generated class (1) and lower recall for the human-written class (0).

